# 📌 Ladder of Thought (LoT)

![Topic](https://img.shields.io/badge/Topic-LoT-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-Sep%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — Small- and medium-sized language models (1 to 8 billion parameters) struggle to perform multi-step reasoning, even after standard fine-tuning on reasoning chains generated by a “teacher.” The “Ladders-of-Thought” (LoT) method addresses this problem by automatically rewriting each training question as a series of progressively easier versions (a “ladder”), classifying these versions into difficulty categories based on the number of reasoning steps they require, and using an adaptive planner to decide, at every moment during training, which difficulty category the model should practice next. (Liu et al., 2026).</span>

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

Large models reason well with chain-of-thought (CoT) prompting, but smaller models often fail to generalize this ability. The usual fix is knowledge distillation: training a small "student" model to imitate a large "teacher" model's step-by-step reasoning and final answers, using a standard next-token prediction loss on the teacher's generated text (Liu et al., 2026). The paper argues this works poorly when the student-teacher capability gap is large, because the student overfits to shallow surface patterns rather than learning transferable reasoning.

A classic remedy is curriculum learning: instead of showing training examples in random order, you show easy examples first and hard ones later, which tends to improve both learning speed and generalization. The obstacle has been defining "difficulty" for reasoning problems in a principled, consistent way, and deciding how to move through the curriculum. LoT addresses both: it derives difficulty directly from the number of reasoning steps a problem needs, and it uses an adaptive (rather than fixed) schedule to move through difficulty levels.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

### 2.1 Progressive Rewrites.

For each training question `q` with a teacher-generated CoT solution `s = (c1, c2, ..., cn)`, the paper treats the question as a set of premises `P = {p1, ..., pm}`. Each reasoning step in the solution derives a conclusion `ci` from a small subset of prior facts `Ai` (premises and/or earlier conclusions). LoT rewrites the question by literally deleting the premises used in a step (Ai) and replacing them with the derived conclusion `ci`. 

Doing this repeatedly produces a chain of versions `q(0), q(1), ..., q(n)` of strictly decreasing difficulty, because each rewrite hands the model a partially pre-computed inference instead of making it derive that fact itself (Liu et al., 2026, Section 3.1). This is generated in practice by prompting a capable instruction-tuned LLM to identify which premises belong to each step and to produce a simplified but well-posed, semantically equivalent question; the rewriting model does not have to be the same model used as the CoT teacher, although in all main experiments GPT-5-mini plays both roles (Liu et al., 2026). Importantly, the paper distinguishes this from decomposition methods (like LADDER, Simonds and Yoshiyama, 2025) that spin off separate sub-problems; LoT instead keeps rewriting the same question, just with some inferences already folded into the premises.

To check that these rewrites are trustworthy, the authors ran a 500-sample audit judged by GPT-5 on three criteria: whether the rewritten question is valid and solvable, whether it is genuinely easier, and whether its answer still matches the original. They report 99.2% of rewrites were valid, 98.0% showed a clear difficulty decrease, and 98.6% preserved the original answer (Liu et al., 2026, Appendix A.2). Note that the judge (GPT-5) belongs to the same model family as the rewriter (GPT-5-mini), which may bias the audit favorably.

### 2.2 Step-Based Difficulty Labeling.

Each rewritten version gets a difficulty score `φ(x)`, defined as the model-estimated minimal number of reasoning steps still required to solve it. Because each rewrite removes exactly one reasoning step by construction, φ(x) decreases by one at each stage of the ladder (e.g., 3 → 2 → 1 → 0), giving a clean, monotonic difficulty measure rather than a noisy proxy like raw CoT text length (Liu et al., 2026, Section 3.2). Training examples are then grouped into buckets `Bk = {x : φ(x) ∈ Ik}`. In the actual implementation, questions needing 0, 1, 2, or 3 steps each get their own bucket, and everything needing 4 or more steps is merged into a single "4+" bucket, to avoid fragmenting the sparse long tail of hard examples (Liu et al., 2026, Appendix A.3.3).

### 2.3 Self-Evolving Curriculum via a Multi-Armed Bandit.

A multi-armed bandit (MAB) is a decision framework where an agent repeatedly chooses among several options ("arms"), each of which returns some payoff, and it must balance exploitation (using the arm that has worked well so far) against exploration (trying arms it knows less about, in case they are actually better). Here, each difficulty bucket is treated as one arm (Liu et al., 2026, Section 3.3, following Matiisen et al., 2019). At each training step, the scheduler picks a bucket, trains the student on a mini-batch from it using the standard CoT distillation loss, and periodically (every `m` steps: 50 for math, 10 for multi-hop) evaluates the model on a held-out validation set that draws the same number of items from every bucket (Algorithm 3). Rewards and Q-values are then computed for every bucket, not only the one just trained on. The reward signal, i.e. the numeric feedback used to judge how good a choice was, is defined as the model's accuracy gain on a bucket relative to a running baseline for that bucket, where the baseline is itself an exponential moving average (a smoothed running average that weights recent observations more heavily) of past accuracy on that bucket. This reward updates a Q-value (an estimate of how valuable each bucket currently is) for each bucket via `Qt+1(a) = α · rt(a) + (1-α) · Qt(a)`. Buckets are then sampled either from a Boltzmann (softmax-style) distribution over Q-values, or via an epsilon-greedy rule that usually exploits the best-known bucket but occasionally explores at random. Because the Q-values, and therefore the sampling probabilities, keep updating as the student's performance changes, the curriculum is "self-evolving" rather than a fixed easy-to-hard schedule (Liu et al., 2026, Algorithm 2).


![LoT.png](../assets/LoT.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Step-based difficulty measure** | Rewrites are labeled by minimal reasoning-step count, a formal, monotonic signal instead of noisy proxies like CoT length or model hit-rate; the paper's own ablation shows this beats a success-rate curriculum|
| 🟢 | **Same-question rewrites, not decomposition** | Each rewrite keeps the original question's identity by folding solved premises into it, rather than spawning separate sub-problems like LADDER does, so the student practices the same task with some inferences pre-computed (the paper does not directly compare against decomposition) |
| 🟢 | **Self-evolving bandit scheduler** | Adaptive bucket sampling converges faster and reaches higher average math accuracy (42.55%) than flat/random (37.60%), easy-to-hard (40.94%), or hard-to-easy (20.31%) fixed schedules (OPT-1.3B, average over 5 math benchmarks, Table 12) |
| 🟢 | **Low added training cost** |Rewrite generation happens once, offline (about $15 for GSM8K and $7 for EntailmentBank with the GPT-5-mini batch API); the bandit scheduler itself adds only 3-8% to total training wall-clock time |
| 🟢 | **Generalizes across model scales and families** | Gains mostly hold from OPT-1.3B/2.7B and Pythia-1.4B/2.8B up through Qwen2.5-7B and Llama3.1-8B (e.g., Llama3.1-8B AddSub +31.19pp, SVAMP +10.37pp vs CoT KD), with a few small regressions (e.g., Qwen2.5-7B ASDiv -2.75pp, Pythia-2.8B MultiArith -1.66pp) |
| 🔴 | **Dataset-dependent gains on multi-hop reasoning** | QASC improves for every model and StrategyQA for 3 of 4 (OPT-2.7B -2.8pp), but MuSiQue and OpenBookQA sometimes regress (e.g., OPT-2.7B MuSiQue drops -19.6pp vs CoT KD) |
| 🔴 | **Sensitive to rewrite depth** | Using all rewritten variants underperforms a moderate depth; "All" depth shows a -3.54pp average drop in math accuracy vs depth ≤3 (OPT-1.3B), even though the main OPT-1.3B results use all depths |
| 🔴 | **Dependent on rewriter model quality** | With a Qwen2.5-7B student and a fixed GPT-5-mini teacher, GSM8K accuracy is 74.60% with GPT-5-mini rewrites but only 25.70% / 19.94% / 11.75% with Qwen2.5-7B / 14B / 72B-Instruct rewriters: quality does not follow rewriter size, and all three fall well below plain CoT KD (73.84%) on GSM8K |
| 🔴 | **Narrow evaluation scope** | Tested only on English, text-only math and multi-hop QA with 1-8B models; scaling to larger models, other languages, or modalities remains untested |
| 🔴 | **Residual rewrite noise unquantified** | The 500-sample audit found 98.6% answer preservation and 98.0% clear difficulty decrease, meaning a small fraction of imperfect rewrites persist, and their downstream impact isn't measured; the judge (GPT-5) is also from the same family as the rewriter |

> All accuracies above are **pass@5** (5 samples at temperature 0.5, counted correct if any passes verification); pass@1 results are only reported in the paper's appendix.


---
## 4. Code Example

> **Goal:** Toy simulation of the Ladders-of-Thought self-evolving scheduler (Algorithms 2-3). Without any hard-coded order, the bandit discovers an easy-to-hard curriculum: it moves to the next bucket once the previous one is mastered, because that is where accuracy gains become largest.

In [1]:
"""
Toy simulation of the Ladders-of-Thought idea (Liu et al., 2026, Algorithms 2-3):
1. Simulate "progressive rewrites" as pre-made difficulty ladders per question.
2. Bucket by step count (0, 1, 2, 3 and a merged "4+" bucket, as in Appendix A.3.3).
3. Run a non-stationary multi-armed bandit scheduler that picks which bucket
   to "train on" next. Every m steps, ALL buckets are evaluated on a balanced
   validation set, and each bucket's Q-value is updated with its accuracy gain
   over an EMA baseline.

No real model calls are made; "training" and "accuracy" are simulated.
"""

import math
import random

random.seed(0)

# --- Step 1 & 2: simulate a dataset already organized into difficulty buckets ---
# Keys are step counts; bucket 4 stands for the merged "4+" bucket (hardest)
buckets = {
    0: [f"trivial_q_{i}" for i in range(20)],
    1: [f"easy_q_{i}" for i in range(20)],
    2: [f"medium_q_{i}" for i in range(20)],
    3: [f"hard_q_{i}" for i in range(20)],
    4: [f"very_hard_q_{i}" for i in range(20)],
}

# Hidden "skill" of the simulated student on each bucket (= its true accuracy)
skill = {0: 0.30, 1: 0.15, 2: 0.05, 3: 0.02, 4: 0.01}


def simulate_train_step(bucket_id):
    """
    Fake a training step on one bucket. Learning a bucket is faster once the
    bucket just below it is mastered (easier reasoning patterns are prerequisites),
    and a little skill transfers to adjacent buckets.
    """
    q = random.choice(buckets[bucket_id])
    readiness = 1.0 if bucket_id == 0 else 0.1 + 0.9 * skill[bucket_id - 1]
    skill[bucket_id] += 0.03 * readiness * (1 - skill[bucket_id])
    for neighbor in (bucket_id - 1, bucket_id + 1):
        if neighbor in skill:
            skill[neighbor] += 0.003 * (1 - skill[neighbor])
    return q


def validate_balanced():
    """Algorithm 3: evaluate the model on an equal number of items from every bucket."""
    return {b: min(max(skill[b] + random.uniform(-0.01, 0.01), 0.0), 1.0) for b in buckets}


# --- Step 3: bandit scheduler state (Algorithm 2) ---
alpha = 0.5          # Q-value learning rate
beta = 0.3           # EMA smoothing for baseline accuracy
tau = 0.02           # Boltzmann temperature
epsilon = 0.1        # exploration rate for epsilon-greedy
policy = "boltzmann" # or "epsilon_greedy"

Q = {b: 0.0 for b in buckets}   # Q-values per bucket (arm), Q_0(a) = 0
baseline_acc = validate_balanced()  # EMA baseline, initialized with a first validation


def choose_bucket():
    """Boltzmann or epsilon-greedy arm selection."""
    if policy == "boltzmann":
        weights = [math.exp(Q[b] / tau) for b in buckets]
        return random.choices(list(buckets), weights=weights)[0]
    if random.random() < epsilon:
        return random.choice(list(buckets))
    return max(Q, key=Q.get)


def update_bandit():
    """Validate every bucket, compute reward as gain over EMA baseline, update all Q-values."""
    acc = validate_balanced()
    for b in buckets:
        reward = acc[b] - baseline_acc[b]
        Q[b] = alpha * reward + (1 - alpha) * Q[b]
        baseline_acc[b] = (1 - beta) * baseline_acc[b] + beta * acc[b]
    return acc


# --- Training loop ---
validate_every = 10  # m
window = 100
counts = {b: 0 for b in buckets}

print("steps      | share of mini-batches per bucket (0, 1, 2, 3, 4+) | val acc per bucket")
for step in range(1, 601):
    bucket_id = choose_bucket()
    simulate_train_step(bucket_id)
    counts[bucket_id] += 1

    if step % validate_every == 0:
        acc = update_bandit()

    if step % window == 0:
        shares = " ".join(f"{counts[b] / window:.2f}" for b in buckets)
        accs = " ".join(f"{acc[b]:.2f}" for b in buckets)
        print(f"{step - window + 1:>3}-{step:<3}    | {shares:<49} | {accs}")
        counts = {b: 0 for b in buckets}

print("\nFinal Q-values (recent learning progress per bucket):")
for b, q_value in Q.items():
    label = "4+" if b == 4 else b
    print(f"  bucket {label}: Q = {q_value:.4f}")

steps      | share of mini-batches per bucket (0, 1, 2, 3, 4+) | val acc per bucket
  1-100    | 0.70 0.01 0.12 0.09 0.08                          | 0.92 0.34 0.17 0.12 0.08
101-200    | 0.48 0.51 0.00 0.00 0.01                          | 0.99 0.88 0.27 0.12 0.07
201-300    | 0.01 0.58 0.41 0.00 0.00                          | 1.00 0.98 0.81 0.23 0.07
301-400    | 0.00 0.00 0.78 0.22 0.00                          | 0.99 0.99 0.99 0.69 0.13
401-500    | 0.00 0.00 0.00 0.85 0.15                          | 0.98 0.98 0.99 0.97 0.58
501-600    | 0.00 0.00 0.00 0.00 1.00                          | 0.99 0.99 0.99 0.98 0.98

Final Q-values (recent learning progress per bucket):
  bucket 0: Q = 0.0015
  bucket 1: Q = -0.0007
  bucket 2: Q = -0.0009
  bucket 3: Q = 0.0017
  bucket 4+: Q = 0.0904


---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **LoT rewrites each question into an easier version by folding in already-solved reasoning steps, not by inventing new sub-problems.**
- **Difficulty is measured by minimal reasoning-step count, giving a clean, monotonic ladder instead of a noisy proxy.**
- **A multi-armed bandit treats each difficulty bucket as an arm and reallocates training toward buckets where the student is improving fastest.**
- **The reward signal is accuracy gain over a running per-bucket baseline, computed from periodic balanced validation, making the curriculum self-evolving rather than fixed.**
- **Gains are largest on out-of-distribution arithmetic tasks (e.g., +32pp AddSub, +25pp SVAMP on OPT-2.7B, pass@5) but are mixed on some multi-hop benchmarks like MuSiQue and OpenBookQA.**

</div>

---
## 6. Source


Liu, M., Magelinski, T., Yuan, D., Yu, Q. & Huang, F. (2026). *Ladders of Thought: A Self-Evolving Curriculum of Progressively Simplified Reasoning Traces.* arXiv:2609.25643. https://arxiv.org/abs/2609.25643